In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = "/content/drive/MyDrive/Dtata_science/Data_Science_Mate/project2.csv"


df_raw = pd.read_csv(file_path)

print(df_raw.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800996 entries, 0 to 800995
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   date        800996 non-null  object
 1   country     800996 non-null  object
 2   device      800996 non-null  object
 3   continent   800996 non-null  object
 4   channel     800996 non-null  object
 5   test        800996 non-null  int64 
 6   test_group  800996 non-null  int64 
 7   event_name  800996 non-null  object
 8   val_count   800996 non-null  int64 
dtypes: int64(3), object(6)
memory usage: 55.0+ MB
None


In [ ]:
print(df_raw["test"].unique())

[4 3 2 1]


In [ ]:
pivot = (
    df_raw
    .pivot_table(
        index=["test", "test_group"],
        columns="event_name",
        values="val_count",
        aggfunc="sum"
    )
    .reset_index()
    .fillna(0)
)

In [ ]:
metrics = {
    "add_payment_info / session": "add_payment_info",
    "add_shipping_info / session": "add_shipping_info",
    "begin_checkout / session": "begin_checkout",
    "new_accounts / session": "new_account"
}

In [ ]:
print(pivot.columns)

Index(['test', 'test_group', 'add_payment_info', 'add_shipping_info',
       'add_to_cart', 'begin_checkout', 'click', 'first_visit', 'new_account',
       'orders', 'page_view', 'scroll', 'select_item', 'select_promotion',
       'session_count', 'session_start', 'user_engagement', 'view_item',
       'view_item_list', 'view_promotion', 'view_search_results'],
      dtype='object', name='event_name')


In [ ]:
print(pivot[["test", "test_group"]].head(20))
print(df_raw["test_group"].unique())

event_name  test  test_group
0              1           1
1              1           2
2              2           1
3              2           2
4              3           1
5              3           2
6              4           1
7              4           2
[2 1]


In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import pandas as pd

metrics = {
    "add_payment_info / session": "add_payment_info",
    "add_shipping_info / session": "add_shipping_info",
    "begin_checkout / session": "begin_checkout",
    "new_accounts / session": "new_account"
}

results = []

for test_id in sorted(pivot["test"].unique()):

    test_data = pivot[pivot["test"] == test_id]

    control = test_data[test_data["test_group"] == 1]
    variant = test_data[test_data["test_group"] == 2]

    for metric_name, metric_col in metrics.items():

        control_conv = control[metric_col].iloc[0]
        test_conv = variant[metric_col].iloc[0]

        control_sessions = control["session_count"].iloc[0]
        test_sessions = variant["session_count"].iloc[0]

        control_cr = control_conv / control_sessions
        test_cr = test_conv / test_sessions

        uplift = (
            (test_cr - control_cr) / control_cr * 100
        )

        counts = [test_conv, control_conv]
        nobs = [test_sessions, control_sessions]

        z_stat, p_value = proportions_ztest(
            counts,
            nobs
        )

        results.append({
            "test": test_id,
            "metric_name": metric_name,
            "control_cr": round(control_cr, 4),
            "test_cr": round(test_cr, 4),
            "uplift_%": round(uplift, 2),
            "p_value": round(p_value, 6),
            "statistical_significance":
                "Significant"
                if p_value < 0.05
                else "Not Significant"
        })

results_df = pd.DataFrame(results)
results_df

,test,metric_name,control_cr,test_cr,uplift_%,p_value,statistical_significance
0,1,add_payment_info / session,0.0438,0.0493,12.54,0.000087,Significant
1,1,add_shipping_info / session,0.0669,0.0713,6.56,0.009226,Significant
2,1,begin_checkout / session,0.0834,0.0890,6.66,0.002894,Significant
3,1,new_accounts / session,0.0843,0.0815,-3.35,0.122859,Not Significant
4,2,add_payment_info / session,0.0463,0.0479,3.58,0.214608,Not Significant
5,2,add_shipping_info / session,0.0687,0.0699,1.65,0.477979,Not Significant
6,2,begin_checkout / session,0.0842,0.0858,1.99,0.340642,Not Significant
7,2,new_accounts / session,0.0823,0.0833,1.24,0.556000,Not Significant
8,3,add_payment_info / session,0.0517,0.0525,1.47,0.520112,Not Significant
9,3,add_shipping_info / session,0.0756,0.0737,-2.62,0.157442,Not Significant


In [ ]:
results_df["is_significant"] = (
    results_df["p_value"] < 0.05
)

In [ ]:
print(results_df.head)

<bound method NDFrame.head of     test                  metric_name  control_cr  test_cr  uplift_%  \
0      1   add_payment_info / session      0.0438   0.0493     12.54   
1      1  add_shipping_info / session      0.0669   0.0713      6.56   
2      1     begin_checkout / session      0.0834   0.0890      6.66   
3      1       new_accounts / session      0.0843   0.0815     -3.35   
4      2   add_payment_info / session      0.0463   0.0479      3.58   
5      2  add_shipping_info / session      0.0687   0.0699      1.65   
6      2     begin_checkout / session      0.0842   0.0858      1.99   
7      2       new_accounts / session      0.0823   0.0833      1.24   
8      3   add_payment_info / session      0.0517   0.0525      1.47   
9      3  add_shipping_info / session      0.0756   0.0737     -2.62   
10     3     begin_checkout / session      0.1361   0.1315     -3.35   
11     3       new_accounts / session      0.0836   0.0827     -1.13   
12     4   add_payment_info / sess

In [ ]:
results_df.to_csv(
    "ab_test_significance_results.csv",
    index=False
)

Conclusion

The A/B testing analysis was performed for four conversion metrics across four experiments. Statistical significance was evaluated using a two-sided z-test for proportions with a significance level of 0.05.

The results showed that Test 1 produced statistically significant improvements in three out of four metrics, including add payment information, add shipping information, and begin checkout conversions. Therefore, Test 1 can be considered successful.

Test 2 showed no statistically significant differences between the control and test groups for any metric.

Tests 3 and 4 demonstrated statistically significant decreases in some conversion metrics, indicating that the tested changes negatively affected user behavior and should not be implemented.

Overall, the analysis demonstrates how statistical testing can be used to distinguish real effects from random fluctuations and support data-driven product decisions.

Tableau Dashboard:https://public.tableau.com/app/profile/dmytro.ruzhytskyi/viz/affilate_retention_AB_test/Dashboard2

DATASETS HERE:  https://drive.google.com/drive/folders/1mm0oF_KTewXzk7Qm57zJKaqEyKwl4PaZ?usp=sharing



